In [ ]:
# !pip3 install -r ../../requirements.txt

In [ ]:
print("hello")

hello


In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')

print('✅ Biblioteki załadowane')

✅ Biblioteki załadowane


In [ ]:
# ── KONFIGURACJA ────────────────────────────────────────────────────────────────
# Zdefiniuj listę naukowców (ORCID + nazwa wyświetlana).
# Możesz podać do 20 naukowców.

SCIENTISTS = [
    {'orcid': '0000-0002-9969-5257', 'name': 'prof. UAM dr hab. Tomasz Górecki'},           # 114 prac
    {'orcid': '0000-0003-2019-2284', 'name': 'prof. dr hab. Mieczysław Mastyło'},           # 126 prac
    {'orcid': '0000-0002-0742-7694', 'name': 'prof. dr hab. Andrzej Ruciński'},             # 199 prac
    {'orcid': '0000-0002-3517-9034', 'name': 'prof. dr hab. Tomasz Łuczak'},                # 179 prac
    {'orcid': '0000-0002-8311-2117', 'name': 'prof. dr hab. Jerzy Kąkol'},                  # 160 prac
    {'orcid': '0000-0001-8066-4533', 'name': 'prof. UAM dr hab. Filip Graliński'},          #  62 prac
    {'orcid': '0000-0002-2897-3176', 'name': 'prof. UAM dr hab. Krzysztof Dyczkowski'},     #  62 prac
    {'orcid': '0000-0002-9427-3127', 'name': 'prof. dr hab. Tomasz Schoen'},                #  56 prac
    {'orcid': '0000-0003-4520-6451', 'name': 'prof. dr hab. Jerzy Jaworski'},               #  55 prac
    {'orcid': '0000-0002-1648-6987', 'name': 'prof. dr hab. Stanisław Gawiejnowicz'},       #  54 prac
    {'orcid': '0000-0002-2374-346X', 'name': 'dr Wojciech Piotr Pałubicki'},                #  50 prac
    {'orcid': '0000-0001-7122-9206', 'name': 'prof. UAM dr hab. Krzysztof Jassem'},         #  39 prac
    {'orcid': '0000-0002-1186-9612', 'name': 'prof. UAM dr hab. Jacek Marciniak'},          #  29 prac
    {'orcid': '0000-0003-2484-143X', 'name': 'dr Bartosz Naskręcki'},                       #  29 prac
    {'orcid': '0000-0002-4927-9992', 'name': 'dr Dominik Filipiak'},                        #  29 prac
    {'orcid': '0000-0002-2016-2598', 'name': 'dr Marek Kubis'},                             #  28 prac
    {'orcid': '0000-0003-3542-8982', 'name': 'prof. UAM dr hab. Patryk Żywica'},            #  26 prac
]

# Ścieżki do danych
SCIENTISTS_FILE = '../../data/scientists_with_identifiers.csv'
WORKS_FILE      = '../../data/titles_with_abstracts.csv'
EMBEDDINGS_DIR  = '../../embeddings'

# Modele (identyczne jak w pubs_num_stability)
MODELS = {
    'MiniLM-L6':          'all-MiniLM-L6-v2',
    'Specter':             'allenai/specter',
    'MPNet-Base':          'all-mpnet-base-v2',
    'Multilingual-MiniLM': 'paraphrase-multilingual-MiniLM-L12-v2',
    'MiniLM-L12':          'all-MiniLM-L12-v2',
    'BGE-Small':           'BAAI/bge-small-en-v1.5',
}

EMB_TYPE   = 'abstract'   # 'title' lub 'abstract'
TARGET_DIM = 384           # docelowy wymiar po projekcji

# Parametry eksperymentu
SAMPLE_SIZES        = [3, 5, 10, 20, 30, 50, 100, 150]
MAX_COMBOS_PER_SIZE = 300
RANDOM_SEED         = 123

OUTPUT_DIR = 'heatmaps'   # katalog na wyniki
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'📋 Konfiguracja:')
print(f'   Naukowcy ({len(SCIENTISTS)}):')
for s in SCIENTISTS:
    print(f'     • {s["name"]:50s}  ORCID: {s["orcid"]}')
print(f'   Modele:          {list(MODELS.keys())}')
print(f'   Typ embeddingu:  {EMB_TYPE}')
print(f'   Docelowy wymiar: {TARGET_DIM}')
print(f'   Rozmiary próbek: {SAMPLE_SIZES}')
print(f'   Max kombinacji:  {MAX_COMBOS_PER_SIZE}')

📋 Konfiguracja:
   Naukowcy (17):
     • prof. UAM dr hab. Tomasz Górecki                    ORCID: 0000-0002-9969-5257
     • prof. dr hab. Mieczysław Mastyło                    ORCID: 0000-0003-2019-2284
     • prof. dr hab. Andrzej Ruciński                      ORCID: 0000-0002-0742-7694
     • prof. dr hab. Tomasz Łuczak                         ORCID: 0000-0002-3517-9034
     • prof. dr hab. Jerzy Kąkol                           ORCID: 0000-0002-8311-2117
     • prof. UAM dr hab. Filip Graliński                   ORCID: 0000-0001-8066-4533
     • prof. UAM dr hab. Krzysztof Dyczkowski              ORCID: 0000-0002-2897-3176
     • prof. dr hab. Tomasz Schoen                         ORCID: 0000-0002-9427-3127
     • prof. dr hab. Jerzy Jaworski                        ORCID: 0000-0003-4520-6451
     • prof. dr hab. Stanisław Gawiejnowicz                ORCID: 0000-0002-1648-6987
     • dr Wojciech Piotr Pałubicki                         ORCID: 0000-0002-2374-346X
     • prof. UAM dr 

In [ ]:
# ── Wczytanie danych ────────────────────────────────────────────────────────────
df_scientists = pd.read_csv(SCIENTISTS_FILE)
df_works      = pd.read_csv(WORKS_FILE)

def clean_orcid(orcid):
    if pd.isna(orcid):
        return None
    return str(orcid).strip().replace('https://orcid.org/', '').replace('http://orcid.org/', '')

df_scientists['clean_orcid'] = df_scientists['orcid'].apply(clean_orcid)
df_works['clean_orcid']      = df_works['main_author_orcid'].apply(clean_orcid)

# Buduj słownik orcid → lista openalex_id
works_by_orcid = (
    df_works
    .dropna(subset=['clean_orcid', 'openalex_id'])
    .groupby('clean_orcid')['openalex_id']
    .apply(list)
    .to_dict()
)

# Sprawdź dostępność danych dla każdego naukowca
print('✅ Dane wczytane.')
print(f'   Łącznie naukowców: {len(df_scientists)}')
print(f'   Łącznie prac:      {len(df_works)}\n')
print('Prace dostępne dla skonfigurowanych naukowców:')
for s in SCIENTISTS:
    orcid = s['orcid']
    works = works_by_orcid.get(orcid, [])
    status = '✅' if works else '⚠ BRAK'
    print(f'   {status}  {s["name"]:30s}  {len(works):>4} prac  ({orcid})')

✅ Dane wczytane.
   Łącznie naukowców: 164
   Łącznie prac:      3440

Prace dostępne dla skonfigurowanych naukowców:
   ✅  prof. UAM dr hab. Tomasz Górecki   114 prac  (0000-0002-9969-5257)
   ✅  prof. dr hab. Mieczysław Mastyło   126 prac  (0000-0003-2019-2284)
   ✅  prof. dr hab. Andrzej Ruciński   199 prac  (0000-0002-0742-7694)
   ✅  prof. dr hab. Tomasz Łuczak      179 prac  (0000-0002-3517-9034)
   ✅  prof. dr hab. Jerzy Kąkol        160 prac  (0000-0002-8311-2117)
   ✅  prof. UAM dr hab. Filip Graliński    62 prac  (0000-0001-8066-4533)
   ✅  prof. UAM dr hab. Krzysztof Dyczkowski    62 prac  (0000-0002-2897-3176)
   ✅  prof. dr hab. Tomasz Schoen       56 prac  (0000-0002-9427-3127)
   ✅  prof. dr hab. Jerzy Jaworski      55 prac  (0000-0003-4520-6451)
   ✅  prof. dr hab. Stanisław Gawiejnowicz    54 prac  (0000-0002-1648-6987)
   ✅  dr Wojciech Piotr Pałubicki       50 prac  (0000-0002-2374-346X)
   ✅  prof. UAM dr hab. Krzysztof Jassem    39 prac  (0000-0001-7122-9206)
   ✅ 

In [ ]:
# ── Lista wszystkich naukowców posortowana wg liczby prac ──────────────────────
# Pomocnicza komórka do konfiguracji — skopiuj ORCID i nazwisko do listy SCIENTISTS powyżej.

scientist_stats = []
for _, row in df_scientists.iterrows():
    orcid = row['clean_orcid']
    name  = row.get('full_name', row.get('name', str(row.get('orcid', ''))))
    n_works = len(works_by_orcid.get(orcid, []))
    scientist_stats.append({'name': name, 'orcid': orcid, 'n_works': n_works})

df_stats = (pd.DataFrame(scientist_stats)
            .sort_values('n_works', ascending=False)
            .reset_index(drop=True))

print(f'{"#":>4}  {"Nazwisko / Imię":45}  {"ORCID":22}  {"Prac":>5}')
print('─' * 85)
for i, row in df_stats.iterrows():
    print(f'{i+1:>4}  {str(row["name"])[:45]:45}  {str(row["orcid"]):22}  {row["n_works"]:>5}')

   #  Nazwisko / Imię                                ORCID                    Prac
─────────────────────────────────────────────────────────────────────────────────────
   1  prof. dr hab. Andrzej Ruciński                 0000-0002-0742-7694       199
   2  prof. dr hab. Tomasz Łuczak                    0000-0002-3517-9034       179
   3  prof. dr hab. Jerzy Kąkol                      0000-0002-8311-2117       160
   4  prof. dr hab. Mieczysław Mastyło               0000-0003-2019-2284       126
   5  prof. UAM dr hab. Tomasz Górecki               0000-0002-9969-5257       114
   6  prof. dr hab. Roman Murawski                   0000-0002-2392-4869       100
   7  prof. dr hab. Jerzy Kaczorowski                0000-0003-1045-122X        92
   8  Jan Wojciech Nowak                             0009-0001-9764-4798        69
   9  prof. dr hab. Leszek Skrzypczak                0000-0002-7484-2900        69
  10  prof. UAM dr hab. Łukasz Smaga                 0000-0002-2442-8816        66
 

In [ ]:
# ── Funkcje pomocnicze (identyczne z pubs_num_stability) ───────────────────────

def load_work_embedding(openalex_id: str, model_short: str, emb_type: str) -> np.ndarray | None:
    model_norm = MODELS[model_short].replace('/', '-').replace('\\', '-')
    subdir     = 'titles' if emb_type == 'title' else 'abstracts'
    filepath   = os.path.join(EMBEDDINGS_DIR, subdir, model_norm,
                              f'{openalex_id}_{emb_type}.pkl')
    if not os.path.exists(filepath):
        return None
    with open(filepath, 'rb') as f:
        return pickle.load(f)['embedding']


def build_centroid(embeddings: list) -> np.ndarray:
    return np.mean(np.stack(embeddings), axis=0)


def load_all_embeddings_for_scientist(work_ids: list, model_short: str,
                                       emb_type: str) -> dict:
    result = {}
    for oid in work_ids:
        emb = load_work_embedding(oid, model_short, emb_type)
        if emb is not None:
            result[oid] = emb
    return result


_projection_cache: dict = {}

def resize_embedding(embedding: np.ndarray, target_dim: int, seed: int = 42) -> np.ndarray:
    """Losowa projekcja liniowa (Johnson-Lindenstrauss) + normalizacja L2."""
    embedding = np.asarray(embedding, dtype=np.float32)
    input_dim = embedding.shape[0]

    if input_dim == target_dim:
        norm = np.linalg.norm(embedding)
        return embedding / norm if norm > 0 else embedding

    cache_key = (input_dim, target_dim, seed)
    if cache_key not in _projection_cache:
        rng_proj = np.random.RandomState(seed + input_dim * 100_000 + target_dim)
        _projection_cache[cache_key] = (
            rng_proj.randn(input_dim, target_dim).astype(np.float32) / np.sqrt(target_dim)
        )

    projected = embedding @ _projection_cache[cache_key]
    norm = np.linalg.norm(projected)
    return projected / norm if norm > 0 else projected


print('✅ Funkcje pomocnicze gotowe.')

✅ Funkcje pomocnicze gotowe.


In [ ]:
# ── Budowanie centroidów dla wszystkich naukowców × modeli ─────────────────────
#
# Struktura wynikowa:
#   all_data[orcid][model_short] = {
#       'entries': [{'size': int, 'centroid': np.ndarray}, ...],
#       'n_all':   int,
#       'valid_sizes': [int, ...]
#   }

from math import comb as n_combinations

rng = np.random.default_rng(RANDOM_SEED)
all_data = {}

for sci in SCIENTISTS:
    orcid = sci['orcid']
    name  = sci['name']
    work_ids = works_by_orcid.get(orcid, [])

    if not work_ids:
        print(f'⚠  {name}: brak prac — pomijam')
        continue

    valid_sizes   = [s for s in SAMPLE_SIZES if s <= len(work_ids)]
    skipped_sizes = [s for s in SAMPLE_SIZES if s >  len(work_ids)]

    print(f'\n▶ {name}  ({len(work_ids)} prac)')
    if skipped_sizes:
        print(f'   Pominięte rozmiary (> {len(work_ids)}): {skipped_sizes}')

    all_data[orcid] = {}

    for model_short in MODELS:
        emb_map = load_all_embeddings_for_scientist(work_ids, model_short, EMB_TYPE)
        all_ids = list(emb_map.keys())

        if not all_ids:
            print(f'   ⚠  {model_short}: brak embeddingów')
            continue

        raw_dim = next(iter(emb_map.values())).shape[0]
        if raw_dim != TARGET_DIM:
            emb_map = {oid: resize_embedding(emb, TARGET_DIM) for oid, emb in emb_map.items()}

        entries = []

        for size in valid_sizes:
            total    = n_combinations(len(all_ids), size)
            n_take   = min(total, MAX_COMBOS_PER_SIZE)
            seen     = set()
            max_att  = n_take * 20
            attempts = 0

            while len(seen) < n_take and attempts < max_att:
                sampled = tuple(sorted(rng.choice(all_ids, size=size, replace=False).tolist()))
                attempts += 1
                if sampled in seen:
                    continue
                seen.add(sampled)
                entries.append({
                    'size':     size,
                    'centroid': build_centroid([emb_map[oid] for oid in sampled]),
                })

        # Centroid ze wszystkich prac
        entries.append({
            'size':     len(all_ids),
            'centroid': build_centroid([emb_map[oid] for oid in all_ids]),
        })

        all_data[orcid][model_short] = {
            'entries':     entries,
            'n_all':       len(all_ids),
            'valid_sizes': valid_sizes,
        }

    print(f'   ✅ Modeli gotowych: {len(all_data[orcid])}')

print(f'\n✅ Centroidy zbudowane dla {len(all_data)} naukowców.')


▶ prof. UAM dr hab. Tomasz Górecki  (114 prac)
   Pominięte rozmiary (> 114): [150]
   ✅ Modeli gotowych: 6

▶ prof. dr hab. Mieczysław Mastyło  (126 prac)
   Pominięte rozmiary (> 126): [150]
   ✅ Modeli gotowych: 6

▶ prof. dr hab. Andrzej Ruciński  (199 prac)
   ✅ Modeli gotowych: 6

▶ prof. dr hab. Tomasz Łuczak  (179 prac)
   ✅ Modeli gotowych: 6

▶ prof. dr hab. Jerzy Kąkol  (160 prac)
   ✅ Modeli gotowych: 6

▶ prof. UAM dr hab. Filip Graliński  (62 prac)
   Pominięte rozmiary (> 62): [100, 150]
   ✅ Modeli gotowych: 6

▶ prof. UAM dr hab. Krzysztof Dyczkowski  (62 prac)
   Pominięte rozmiary (> 62): [100, 150]
   ✅ Modeli gotowych: 6

▶ prof. dr hab. Tomasz Schoen  (56 prac)
   Pominięte rozmiary (> 56): [100, 150]
   ✅ Modeli gotowych: 6

▶ prof. dr hab. Jerzy Jaworski  (55 prac)
   Pominięte rozmiary (> 55): [100, 150]
   ✅ Modeli gotowych: 6

▶ prof. dr hab. Stanisław Gawiejnowicz  (54 prac)
   Pominięte rozmiary (> 54): [100, 150]
   ✅ Modeli gotowych: 6

▶ dr Wojciech Pio

In [ ]:
# ── Generowanie heatmap — jedna figura na naukowca, 6 paneli (model × model) ───
#
# Każda heatmapa pokazuje macierz podobieństwa cosinusowego między centroidami
# zbudowanymi z różnych rozmiarów próbek prac.
# Oś X = oś Y = rozmiary (pogrupowane: n=3, 5, 10, … , all).
# Każdy naukowiec dostaje osobny plik PNG w OUTPUT_DIR.
# Kwadrat diagonalny z mean cosine sim > 0.85 otrzymuje niebieskie obramowanie.

from matplotlib import patches as mpatches

_cmap = plt.cm.plasma

def size_label(size, n_all):
    return f'all\n(n={size})' if size == n_all else f'n={size}'

def size_color_palette(valid_sizes, n_all):
    """Zwraca {size: hex_color}. 'all' dostaje zielony."""
    palette = {}
    cmap = plt.cm.plasma
    n = len(valid_sizes)
    for i, s in enumerate(valid_sizes):
        palette[s] = cmap(i / max(n - 1, 1))
    palette[n_all] = '#27ae60'
    return palette


for sci in SCIENTISTS:
    orcid = sci['orcid']
    name  = sci['name']

    if orcid not in all_data or not all_data[orcid]:
        print(f'⚠  {name}: brak danych — pomijam')
        continue

    models_avail = list(all_data[orcid].keys())
    n_models     = len(models_avail)
    n_cols       = min(3, n_models)
    n_rows       = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(9 * n_cols, 8 * n_rows),
                              constrained_layout=True)
    axes_flat = np.array(axes).flatten() if n_models > 1 else [axes]

    fig.suptitle(
        f'Heatmapa podobieństwa cosinusowego centroidów\n'
        f'{name}  [{EMB_TYPE}, dim={TARGET_DIM}]',
        fontsize=14, fontweight='bold'
    )

    for ax_i, model_short in enumerate(models_avail):
        ax      = axes_flat[ax_i]
        d       = all_data[orcid][model_short]
        entries = d['entries']
        n_all   = d['n_all']
        vsizes  = d['valid_sizes']
        palette = size_color_palette(vsizes, n_all)

        # Posortuj centroidy wg rozmiaru
        order          = sorted(range(len(entries)), key=lambda i: entries[i]['size'])
        entries_sorted = [entries[i] for i in order]
        sizes_sorted   = [e['size'] for e in entries_sorted]

        vecs    = np.array([e['centroid'] for e in entries_sorted])
        sim_mat = cosine_similarity(vecs)

        im = ax.imshow(sim_mat, cmap='RdYlGn', vmin=0.7, vmax=1.0,
                       aspect='auto', interpolation='nearest')
        plt.colorbar(im, ax=ax, label='cosine sim', shrink=0.85)

        # Etykiety osi — pierwsza pozycja każdego bloku rozmiaru
        tick_positions, tick_labels, tick_colors = [], [], []
        block_info = {}  # size -> (start_idx, end_idx)
        prev = None
        block_start = 0
        for i, s in enumerate(sizes_sorted):
            if s != prev:
                if prev is not None:
                    block_info[prev] = (block_start, i - 1)
                    ax.axhline(i - 0.5, color='white', linewidth=1.5, alpha=0.8)
                    ax.axvline(i - 0.5, color='white', linewidth=1.5, alpha=0.8)
                block_start = i
                tick_positions.append(i)
                tick_labels.append(size_label(s, n_all))
                tick_colors.append(palette.get(s, '#333333'))
                prev = s
        if prev is not None:
            block_info[prev] = (block_start, len(sizes_sorted) - 1)

        ax.set_xticks(tick_positions)
        ax.set_xticklabels(tick_labels, fontsize=8.5)
        ax.set_yticks(tick_positions)
        ax.set_yticklabels(tick_labels, fontsize=8.5)

        for tick, color in zip(ax.get_xticklabels(), tick_colors):
            tick.set_color(color)
        for tick, color in zip(ax.get_yticklabels(), tick_colors):
            tick.set_color(color)

        # ── Obramowanie bloków diagonalnych z mean sim > 0.85 ─────────────────
        BORDER_THRESHOLD = 0.85
        for size, (si, ei) in block_info.items():
            block = sim_mat[si:ei+1, si:ei+1]
            if si == ei:
                mean_sim = block[0, 0]  # jeden centroid — sim=1.0
            else:
                idx = np.triu_indices(ei - si + 1, k=1)
                mean_sim = np.mean(block[idx])
            if mean_sim > BORDER_THRESHOLD:
                width  = ei - si + 1
                rect = mpatches.Rectangle(
                    (si - 0.5, si - 0.5), width, width,
                    linewidth=2.5, edgecolor='#1a6fd4',
                    facecolor='none', zorder=5
                )
                ax.add_patch(rect)

        n_centroids = len(entries)
        ax.set_title(f'{model_short}  ({n_centroids} centroidów, n_all={n_all})', fontsize=11)

    for ax in axes_flat[n_models:]:
        ax.set_visible(False)

    safe_name = name.replace(' ', '_').replace('/', '_')
    out_path  = os.path.join(OUTPUT_DIR, f'heatmap_{safe_name}.png')
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Zapisano: {out_path}')

print('\n✅ Wszystkie heatmapy wygenerowane.')

In [ ]:
# ── Analiza stabilizacji: kiedy mean(cosine sim wewnątrz bloku) > próg ─────────
#
# Dla każdego naukowca × modelu × rozmiaru próbki liczymy:
#   - macierz cosinusową między wszystkimi centroidami z TEGO SAMEGO n
#   - bierzemy górny trójkąt (bez diagonali) → mean i std
# Szukamy najmniejszego n, gdzie mean_sim > THRESHOLD.

THRESHOLD = 0.85

stab_rows = []

for sci in SCIENTISTS:
    orcid = sci['orcid']
    name  = sci['name']
    if orcid not in all_data:
        continue

    for model_short, d in all_data[orcid].items():
        entries    = d['entries']
        n_all      = d['n_all']
        valid_sizes = d['valid_sizes']

        for size in valid_sizes:
            block = [e['centroid'] for e in entries if e['size'] == size]
            if len(block) < 2:
                continue
            mat  = cosine_similarity(np.array(block))
            # górny trójkąt bez diagonali
            idx  = np.triu_indices(len(block), k=1)
            vals = mat[idx]
            stab_rows.append({
                'name':     name,
                'orcid':    orcid,
                'model':    model_short,
                'n':        size,
                'mean_sim': np.mean(vals),
                'std_sim':  np.std(vals),
                'min_sim':  np.min(vals),
            })

df_stab = pd.DataFrame(stab_rows)

# ── Wyznacz punkt stabilizacji dla każdego (naukowiec, model) ─────────────────
stab_points = []
for (name, model), grp in df_stab.groupby(['name', 'model'], sort=False):
    grp_sorted = grp.sort_values('n')
    stable = grp_sorted[grp_sorted['mean_sim'] >= THRESHOLD]
    if stable.empty:
        stab_n = None
    else:
        stab_n = int(stable.iloc[0]['n'])
    stab_points.append({'name': name, 'model': model, 'stab_n': stab_n})

df_sp = pd.DataFrame(stab_points)

# ── Pivot: wiersze = naukowcy, kolumny = modele ───────────────────────────────
model_order = list(MODELS.keys())
pivot = df_sp.pivot(index='name', columns='model', values='stab_n')[model_order]

# Posortuj wg mediany punktu stabilizacji (None → inf)
pivot['_median'] = pivot.apply(
    lambda row: np.median([v for v in row if pd.notna(v)]) if row.notna().any() else float('inf'),
    axis=1
)
pivot = pivot.sort_values('_median').drop(columns='_median')
pivot.index.name   = None
pivot.columns.name = None

print(f'Najmniejsze n, przy którym mean(cosine sim wewnątrz bloku) ≥ {THRESHOLD}')
print(f'(None = próg nie osiągnięty dla żadnego badanego rozmiaru)\n')
print(pivot.to_string(na_rep='—'))

# ── Podsumowanie: ile prac potrzeba na stabilizację (mediana po modelach) ─────
print('\n\nMediana po modelach (prac potrzebnych do stabilizacji):')
print(f'  {"Naukowiec":50s}  {"mediana n":>10}  {"min n":>6}  {"max n":>6}')
print('  ' + '─' * 75)
for name, row in pivot.iterrows():
    vals   = [v for v in row if pd.notna(v)]
    if vals:
        med = np.median(vals)
        mn  = int(min(vals))
        mx  = int(max(vals))
        print(f'  {name[:50]:50s}  {med:>10.0f}  {mn:>6}  {mx:>6}')
    else:
        print(f'  {name[:50]:50s}  {"—":>10}  {"—":>6}  {"—":>6}')

In [ ]:
# ── Podsumowanie: ile prac potrzeba do stabilizacji centroidu (per naukowiec) ──
#
# Dla każdego naukowca pokazujemy:
#   - liczbę prac dostępnych
#   - punkt stabilizacji wg każdego modelu (najmniejsze n z mean_sim > THRESHOLD)
#   - medianę i zakres po modelach

print(f'Stabilizacja centroidu (próg mean cosine sim ≥ {THRESHOLD})\n')
print(f'  {"Naukowiec":<42}  {"Prac":>4}  {"MiniLM-L6":>10}  {"Specter":>8}  {"MPNet":>7}  {"MultiMiniLM":>12}  {"MiniLM-L12":>11}  {"BGE-Small":>10}  {"mediana":>8}')
print('  ' + '─' * 118)

for sci in SCIENTISTS:
    orcid = sci['orcid']
    name  = sci['name']
    n_prac = len(works_by_orcid.get(orcid, []))

    row_vals = []
    per_model = {}
    for model in model_order:
        match = df_sp[(df_sp['name'] == name) & (df_sp['model'] == model)]
        if match.empty:
            per_model[model] = '—'
        else:
            v = match.iloc[0]['stab_n']
            per_model[model] = '—' if pd.isna(v) else int(v)
            if not pd.isna(v):
                row_vals.append(v)

    med_str = f'{np.median(row_vals):.0f}' if row_vals else '—'

    cols = [per_model[m] for m in model_order]
    print(f'  {name[:42]:<42}  {n_prac:>4}  '
          f'{str(cols[0]):>10}  {str(cols[1]):>8}  {str(cols[2]):>7}  '
          f'{str(cols[3]):>12}  {str(cols[4]):>11}  {str(cols[5]):>10}  '
          f'{med_str:>8}')

print(f'\n(—) = próg {THRESHOLD} nie osiągnięty dla żadnego badanego rozmiaru próbki')
print(f'Badane rozmiary próbek: {SAMPLE_SIZES}')